# SAP-RPT — Prediction via the public playground API

The [SAP RPT Playground](https://rpt.cloud.sap) offers a simple, public REST API to make predictions on tabular data. This notebook demonstrates how you can use it with your own dataset.

## About SAP-RPT-1.5

SAP-RPT-1.5 is a table-native foundation model built around **[in-context learning](https://community.sap.com/t5/technology-blog-posts-by-sap/a-new-paradigm-for-enterprise-ai-in-context-learning-for-relational-data/ba-p/14260221)**, separating the input rows into two different categories:

- **Context rows**: labelled rows you supply as examples; the model learns from them at inference time.
- **Query rows**: rows where the target value is unknown and should be predicted.

Rather than training a separate model for each problem, you provide context rows with the target
column filled in, along with query rows where the target is marked for prediction. The model
predicts the marked values using the context rows as examples. No training loop, no model artifact
to manage, only a single API call.

This approach has three practical advantages over classical ML:

- **Instant iteration**: update your context rows and call the API again; results reflect the
  change immediately with no retraining wait.
- **No lifecycle overhead**: there is no trained model to version, deploy, or monitor.
- **No tuning required**: the model reaches high accuracy without hyperparameter search.

The model supports both regression and classification and detects the task type automatically from
the target column.

| Task | Target column | Example in this notebook |
|------|---------------|--------------------------|
| Regression | Numeric values | `Days Late` - the number of days an invoice is paid late |
| Classification | Categories or labels | `Is Late` - whether an invoice is paid late at all |

Both examples use one SAP scenario dataset, `payment_delay_prediction`. 

You can find out more about SAP-RPT-1.5 [here](https://community.sap.com/t5/artificial-intelligence-blogs-posts/sap-rpt-1-5-now-live-on-generative-ai-hub/ba-p/14447819).

**Disclaimers**
- The SAP-RPT-1.5 model can process up to 2,048 context rows per API call. When you send in more context rows, the public API will do a random downsampling to only 2,048 context rows to be sent to the model, and the response reports this under `samplingMetadata`. 
- If you are planning to loop the API call for benchmarking purposes, you can pass `sampler_options.seed` to make the random selection deterministic and looped calls will produce the same downsampled subset if you specified the same seed.
- Note that the SAP-RPT-1.5-large model on SAP GenAI Hub can process more context rows and provides higher accuracy. You can learn more about it [here](https://help.sap.com/docs/sap-ai-core/generative-ai/sap-rpt-1-5?locale=en-US&ai=true). 
- Do note that benchmarking done using the public API may not be 100% representative of our model variants available on SAP generative AI hub. The public API serves as a proxy so there may be additional latency when using this API. 

## 1 · Setup

Install and import the required libraries. The demo depends on `requests` for the API call,
`pandas` and `pyarrow` for data handling, and `python-dotenv` to load the API
token from a `.env` file.

In [ ]:
%pip install -q requests pandas python-dotenv pyarrow

import json
import os

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()  # loads RPT_API_TOKEN from .env if present; shell env vars take priority
pd.set_option("display.max_columns", 30)
print("Libraries ready.")

## 2 · Authentication

The API authenticates with a personal token sent in the `Authorization: Bearer` header. Create a
token on the **Settings** page of [RPT Playground](https://rpt.cloud.sap/) and store it in a
`.env` file in the same directory as this notebook.

Add this line to your `.env` file: `RPT_API_TOKEN="your-token-here"`

The setup
cell calls `load_dotenv()`, which reads `.env` automatically. You can also set `RPT_API_TOKEN` as
a shell environment variable before launching Jupyter; that takes priority over the file.

In [ ]:
API_URL = "https://rpt.cloud.sap/api/predict"


def get_token():
    token = os.environ.get("RPT_API_TOKEN")
    if not token:
        raise RuntimeError(
            "No API token found. Create one on the Settings page at RPT Playground "
            "(https://rpt.cloud.sap/), then either set it as the environment variable "
            "RPT_API_TOKEN or add it to a .env file in the notebook directory."
        )
    return token


API_TOKEN = get_token()
print(f"Token loaded ({len(API_TOKEN)} chars) — not printing it.")

## 3 · The `predict()` function

`predict()` builds the request payload and calls `POST /api/predict`. It takes three required
inputs: the labelled `context_df`, the `query_df` to be answered, and the name of the `target`
column. An optional `index_column` identifies each row in the response; when omitted, predictions
are aligned to the query rows by position.

The payload follows the [API reference](https://rpt.cloud.sap/docs). A single `rows` array holds
the context rows followed by the query rows, with the target column set to the placeholder string
`"[PREDICT]"` on the query rows. The request also sets a fixed `sampler_options.seed` for
reproducibility and `explanations` to return feature importances and the most relevant context rows
for each prediction. The API also accepts an optional `scenario` string (up to 100 characters) to
tag the request; this demo does not send one.

The two `explanations` controls, `top_column_scores` and `top_relevant_context_rows`, each accept a
value from 1 to 20 and default to 0 (disabled). This notebook overrides them to 5 and 5.

`predictions_to_df()` flattens the response into a DataFrame with one row per prediction. In the
response, `prediction.predictions` is a list of row objects, each keyed by the index column and the
target column. The target column holds a single-element list whose object carries `prediction`,
`confidence`, and `confidence_interval`. Regression returns a `confidence_interval` and no
`confidence`; classification returns a `confidence` and no `confidence_interval`. The helper drops
whichever of these columns is entirely empty.

In [ ]:
PREDICT_PLACEHOLDER = "[PREDICT]"


def _clean(df):
    """Replace NaN with empty strings so the payload is valid JSON.

    Pandas loads blank cells as NaN (a float), and strict JSON has no NaN - sending it
    raises 'Out of range float values are not JSON compliant'. Empty string is also how
    this scenario represents missing values, so RPT reads it the same way."""
    return df.astype(object).where(df.notna(), "")


def predict(
    context_df,
    query_df,
    target,
    index_column=None,
    seed=42,
    top_columns=5,
    top_context_rows=5,
):
    """Send context + query rows to /api/predict and return the parsed JSON response."""
    context_rows = _clean(context_df).to_dict(orient="records")
    q = _clean(query_df).copy()
    q[target] = PREDICT_PLACEHOLDER
    query_rows = q.to_dict(orient="records")

    payload = {
        "rows": context_rows + query_rows,
        "sampler_options": {"seed": seed},
        "explanations": {
            "top_column_scores": top_columns,
            "top_relevant_context_rows": top_context_rows,
        },
    }
    if index_column is not None:
        payload["index_column"] = index_column

    assert 1 <= len(query_rows) <= 512, "query rows must be 1..512"
    assert 2 <= len(context_rows) <= 100_000, "context rows must be 2..100000"
    assert len(context_df.columns) <= 100, "max 100 columns"

    headers = {
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json",
    }
    resp = requests.post(API_URL, headers=headers, json=payload, timeout=120)
    resp.raise_for_status()
    return payload, resp.json()


def predictions_to_df(response, target, index_column=None):
    """Flatten a /api/predict response into a tidy DataFrame, one row per prediction.

    When index_column is None, rows are identified by their position in the query set."""
    rows = []
    for i, item in enumerate(response["prediction"]["predictions"]):
        pred = item[target][0]
        row = {
            "predicted": pred.get("prediction"),
            "confidence": pred.get("confidence"),
            "confidence_interval": pred.get("confidence_interval"),
        }
        if index_column is not None:
            row = {index_column: item.get(index_column), **row}
        else:
            row = {"query_row": i, **row}
        rows.append(row)
    return pd.DataFrame(rows).dropna(axis=1, how="all")


print("predict() and predictions_to_df() defined.")

## 4 · Load the example dataset

The `payment_delay_prediction` scenario contains 589 invoices, one per row, across 22 columns that
describe the customer, the invoice, and the payment history. The target column is `Days Late`.

The file ships with four rows whose `Days Late` value is already set to `"[PREDICT]"`. These are the
scenario's own demo queries, whose true value is unknown. Because there is no ground truth for those
rows, they cannot be used to measure the model's accuracy, so this section sets them aside and works
only with the 585 fully labelled rows. Sections 5 and 6 then hold out labelled rows as queries,
which lets the predictions be checked against known values. The four unlabelled rows are predicted
in section 9.

The cell reports the class balance after removing those rows. Most invoices in this scenario are
paid late, which is relevant to the classification example that follows.

In [ ]:
df = pd.read_csv("data/payment_delay_prediction.csv")

# The scenario file ships 4 rows whose Days Late is already "[PREDICT]" (its own demo queries).
# We drop those so every row we work with is fully labelled.
labelled = df[df["Days Late"] != "[PREDICT]"].copy()
labelled["Days Late"] = labelled["Days Late"].astype(float)

print(f"{len(df)} rows total, {len(labelled)} fully labelled.")

# Most invoices in this scenario are paid late - worth knowing before we frame classification.
n_late = int((labelled["Days Late"] > 0).sum())
print(f"Paid late: {n_late}  |  Paid on time: {len(labelled) - n_late}")
labelled.head()

## 5 · Regression — predict the number of days late

A small set of invoices is held out as query rows and the remainder is passed as context. The model
predicts `Days Late` for the query rows. Because the target is numeric, the model treats the task as
regression and returns a predicted value together with a confidence interval.

The query set is chosen deterministically: the labelled rows are sorted by `Days Late`, and the
three earliest and three latest invoices are selected. This produces a query set that spans both
ends of the range, so the predictions can be compared against a spread of actual values.

In [ ]:
INDEX = "Transaction ID"
TARGET_REG = "Days Late"

# Build a small, varied query set: a few on-time invoices plus a few late ones, so both the
# regression spread and the classification example below show more than one outcome. The rest
# of the table becomes context. (Deterministic - we sort by Days Late and pick from both ends.)
by_days = labelled.sort_values("Days Late")
query_reg = pd.concat(
    [by_days.head(3), by_days.tail(3)]
).copy()  # 3 earliest + 3 latest
context_reg = labelled.drop(index=query_reg.index).copy()  # everything else

payload_reg, resp_reg = predict(
    context_reg, query_reg, target=TARGET_REG, index_column=INDEX
)

# Show the predictions next to the true values we held out.
predictions_to_df(resp_reg, TARGET_REG, INDEX).merge(
    query_reg[[INDEX, TARGET_REG]].rename(columns={TARGET_REG: "actual"}),
    on=INDEX,
)

## 6 · Classification — predict whether an invoice will be late

The same data is reframed as a classification task. A categorical target `Is Late` is derived as
`Days Late > 0`. With a label as the target, the model switches to classification and returns
a predicted class together with a confidence score.

The classes in this scenario are imbalanced, since most invoices are paid late. The query set reuses
the split from the regression example, which mixes on-time and late invoices so that both outcomes
appear. Section 8 shows how to predict both targets in a single call.

In [ ]:
TARGET_CLF = "Is Late"


def make_clf_frame(frame):
    f = frame.copy()
    f[TARGET_CLF] = (f["Days Late"] > 0).map({True: "Late", False: "On time"})
    return f.drop(columns=["Days Late"])  # avoid leaking the numeric answer


context_clf = make_clf_frame(context_reg)
query_clf = make_clf_frame(query_reg)

payload_clf, resp_clf = predict(
    context_clf, query_clf, target=TARGET_CLF, index_column=INDEX
)

# Same tidy-up as regression; here the model returns a class label plus a confidence.
predictions_to_df(resp_clf, TARGET_CLF, INDEX).merge(
    query_clf[[INDEX, TARGET_CLF]].rename(columns={TARGET_CLF: "actual"}),
    on=INDEX,
)

## 7 · Explanations — relevant context rows

Every call through the `predict()` helper requests explanations. The most distinctive feature is
`top_relevant_context_rows`: for each query row, the API returns the indices of the context rows
that most influenced the prediction. Looking up those rows shows concretely *why* the model
answered the way it did.

Explanations are returned under `response["prediction"]["explanations"]["top_relevant_context_rows"]`
as a list with one entry per query row. Each entry is a list of integer positions into the
original context array, so `context_reg.iloc[idx]` gives the actual row.

The cell below prints the top relevant context rows for the first three query predictions.
`top_column_scores` (feature importances per column) is also available in the response under the
same `explanations` key; see the [API reference](https://rpt.cloud.sap/docs) for the full schema.

In [ ]:
explanations = resp_reg.get("prediction", {}).get("explanations", {})
relevant_rows = explanations.get("top_relevant_context_rows")

if not relevant_rows:
    print("top_relevant_context_rows not found — raw explanations object:")
    print(json.dumps(explanations, indent=2)[:1000])
else:
    preds = resp_reg["prediction"]["predictions"]
    for i in range(min(3, len(preds))):
        query_id = preds[i].get(INDEX, f"query row {i}")
        predicted = preds[i][TARGET_REG][0]["prediction"]
        indices = relevant_rows[i]  # list of integer positions into context_reg

        print(f"Query: {query_id}  |  Predicted Days Late: {predicted:.1f}")
        print(f"  Top {len(indices)} relevant context rows:")
        display(
            context_reg.iloc[indices][
                [
                    INDEX,
                    TARGET_REG,
                    "Risk Score",
                    "Avg Days Late",
                    "Previous Delays",
                    "Outstanding",
                ]
            ].reset_index(drop=True)
        )
        print()

## 8 · Combined call — regression and classification in one request

Both targets can be predicted in a single API call. `Days Late` (regression) and `Is Late`
(classification) are included together; the API predicts each target independently with no leakage
between them.

The `predict()` helper above stamps `[PREDICT]` on a single target column. For a multi-target call
we build the payload directly, which also makes the request structure explicit.

In [ ]:
# Build context and query frames that contain both target columns.
context_combined = context_reg.copy()
context_combined[TARGET_CLF] = (context_combined[TARGET_REG] > 0).map(
    {True: "Late", False: "On time"}
)

query_combined = query_reg.copy()
query_combined[TARGET_CLF] = (query_combined[TARGET_REG] > 0).map(
    {True: "Late", False: "On time"}
)

# Stamp [PREDICT] on both target columns for the query rows.
q_combined = _clean(query_combined).copy()
q_combined[TARGET_REG] = PREDICT_PLACEHOLDER
q_combined[TARGET_CLF] = PREDICT_PLACEHOLDER

payload_combined = {
    "rows": _clean(context_combined).to_dict(orient="records")
    + q_combined.to_dict(orient="records"),
    "index_column": INDEX,
    "sampler_options": {"seed": 42},
}

# Headers are constructed here directly because we're bypassing the predict() helper.
headers_combined = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}
resp_combined = requests.post(
    API_URL, headers=headers_combined, json=payload_combined, timeout=120
)
resp_combined.raise_for_status()
resp_combined = resp_combined.json()

# Flatten each target into its own DataFrame and merge for display.
df_reg = predictions_to_df(resp_combined, TARGET_REG, INDEX)
df_clf = predictions_to_df(resp_combined, TARGET_CLF, INDEX)

df_reg.merge(df_clf, on=INDEX, suffixes=("_days_late", "_is_late")).merge(
    query_reg[[INDEX, TARGET_REG]].rename(columns={TARGET_REG: "actual_days_late"}),
    on=INDEX,
)

## 9 · Use your own data

The cell below is the template for running the model on any table. It reads a CSV or Parquet file,
sends every labelled row as context, and predicts a query set. To use it on your own data, edit the three values at the top: the path to
your file, the name of the column you want predicted, and a column that uniquely identifies each
row. The model determines whether the task is regression or classification from your target column.

As shipped, it points back at `payment_delay_prediction` to close the loop: section 4 set aside the
four rows whose `Days Late` is marked `"[PREDICT]"` because their true value is unknown, and this is
where those genuinely unknown rows are predicted. The cell adapts to your file in one of two ways:

- **If the target column marks rows with `"[PREDICT]"`** the model predicts exactly those rows. There is no `actual` column,
  because there is no ground truth to compare against.
- **Otherwise** the cell holds out the last five rows as a query set, keeping their true values, and
  shows `predicted` next to `actual` — the same validation view as the regression and classification
  examples above.

In [ ]:
# ── EDIT THESE THREE LINES ────────────────────────────────────────────────
MY_FILE = "data/payment_delay_prediction.csv"  # <- your file (.csv or .parquet)
MY_TARGET = "Days Late"  # <- column to predict
MY_INDEX = "Transaction ID"  # <- a unique-id column
# ──────────────────────────────────────────────────────────────────────────

ext = os.path.splitext(MY_FILE)[1].lower()
if ext == ".parquet":
    mydf = pd.read_parquet(MY_FILE)
elif ext == ".csv":
    # sep=None + engine="python" sniffs the delimiter, so comma- and semicolon-separated
    # exports both load without editing this cell.
    mydf = pd.read_csv(MY_FILE, sep=None, engine="python")
else:
    raise ValueError(
        f"Unsupported file extension '{ext}'. Use a .csv or .parquet file."
    )

# Fail early with a readable message if the two column names above aren't in the file.
missing = [c for c in (MY_TARGET, MY_INDEX) if c not in mydf.columns]
if missing:
    raise KeyError(
        f"Column(s) {missing} not found in {MY_FILE}. "
        f"Available columns: {list(mydf.columns)}"
    )

# Two modes, chosen automatically:
#   1. Rows marked "[PREDICT]" are genuine unknowns — predict exactly those (no actual to show).
#   2. No markers — hold out the last 5 rows as a query set and compare predicted vs actual.
predict_mask = mydf[MY_TARGET].astype(str) == PREDICT_PLACEHOLDER

if predict_mask.any():
    my_query = mydf[predict_mask].copy()  # the rows you want answered
    my_context = mydf[~predict_mask].copy()  # everything labelled becomes examples
    have_actual = False
    print(f"Found {len(my_query)} row(s) marked [PREDICT] — predicting those.")
else:
    my_query = mydf.tail(5).copy()  # hold out the last 5 rows
    my_context = mydf.iloc[:-5].copy()  # everything else as examples
    have_actual = True
    print(
        f"No [PREDICT] rows — holding out the last {len(my_query)} row(s) to check against actuals."
    )

my_payload, my_resp = predict(
    my_context, my_query, target=MY_TARGET, index_column=MY_INDEX
)

print(f"Sent {len(my_payload['rows'])} rows to the model.")

out = predictions_to_df(my_resp, MY_TARGET, MY_INDEX)
if have_actual:
    # We kept the true values, so show them next to the predictions (as in sections 5 & 6).
    out = out.merge(
        my_query[[MY_INDEX, MY_TARGET]].rename(columns={MY_TARGET: "actual"}),
        on=MY_INDEX,
    )
out

## Limits and next steps

You can refer to our playground documentation for the API limits and specifications: [rpt.cloud.sap/docs/api/overview](https://rpt.cloud.sap/docs/api/overview)

**Next steps**

- Apply `predict()` to your own data by swapping in your file path, target column, and index column.
- Predict several targets in one call, as shown in section 8.
- Try the built-in scenarios interactively on [RPT Playground](https://rpt.cloud.sap).